Yes — that is exactly the right question.

A service like **PostgreSQL** or **GitLab** can be installed and managed on Linux in **different packaging/service models**, so it may appear in `ps`, `systemctl`, both, or neither in the way you expect.

## The simple idea

There are two separate things:

* **How software is installed/packaged**
* **How its processes are supervised/managed**

That is why the same DB can show up differently on different machines.

---

## Common ways software like PostgreSQL/GitLab can exist on Linux

### 1. Native OS package + systemd

This is the most common normal Linux way.

Example:

* installed with `apt install postgresql`
* managed by `systemctl`

You will usually see:

```bash
systemctl status postgresql
ps aux | grep postgres
```

Files often live in places like:

* binaries: `/usr/bin/`
* config: `/etc/postgresql/` or `/etc/mysql/`
* data: `/var/lib/postgresql/` or `/var/lib/mysql/`

This is the usual **distro-managed install**.

---

### 2. Bundled application install with its own embedded services

This is what **GitLab Omnibus** does.

GitLab installs:

* its **own PostgreSQL**
* its **own Redis**
* its **own Nginx**
* its **own monitoring/exporters**
* and manages them itself

So instead of OS packages like:

* `/usr/bin/postgres`
* `postgresql.service`

you may see:

* `/opt/gitlab/embedded/bin/postgres`
* managed by `gitlab-ctl`

This is why:

* `gitlab-ctl status` shows GitLab components
* `systemctl` may only show a top-level GitLab-related unit, or sometimes not much
* `ps aux` still shows the actual processes, but under GitLab paths

In your case, this is why you saw:

```bash
/opt/gitlab/embedded/bin/postgres
/opt/gitlab/embedded/bin/redis-server
```

So GitLab components are often installed as:

* **embedded binaries under `/opt/gitlab/embedded/bin/`**
* configs/data/logs under `/etc/gitlab`, `/var/opt/gitlab`, `/var/log/gitlab`
* supervised internally by **runit**, exposed through `gitlab-ctl`

---

### 3. Containerized

The DB/service runs in Docker or Podman.

You may not see a normal host service name. Instead you check:

```bash
docker ps
podman ps
```

and maybe:

```bash
ps aux | grep docker
```

A PostgreSQL container may expose port 5432, or may only be reachable inside container networking.

---

### 4. Kubernetes

The DB/app runs as pods, not as normal host services.

You won’t inspect it mainly with `systemctl` or `ps aux` on the host. Instead:

```bash
kubectl get pods -A
kubectl get svc -A
```

---

### 5. Manual / source install

Someone compiled or unpacked it manually and starts it with scripts.

Example:

* custom binary in `/opt/postgres/bin/postgres`
* started by shell script, cron, tmux, screen, rc.local, nohup, etc.

Then:

* `ps aux` may show it
* `systemctl` may show nothing

---

## In your GitLab case, what is `gitlab-ctl` using?

GitLab Omnibus commonly uses **runit** internally.

That is why in your earlier output you saw processes like:

* `runsv postgresql`
* `svlogd ...`

Those are strong signs of **runit supervision**.

So the model is roughly:

* **systemd** may start the top-level GitLab service/package
* **gitlab-ctl** talks to GitLab’s internal supervisor
* **runit** manages individual components like PostgreSQL, Redis, Puma, Sidekiq, etc.

So GitLab components are not plain `postgresql.service`, `redis.service`, etc. They are often **GitLab-managed embedded services**.

---

## Why `gitlab-runner` showed under `systemctl`

Because `gitlab-runner` is often installed separately as a **normal Linux service**.

So it can appear like this:

```bash
systemctl status gitlab-runner
```

That does **not** mean the GitLab app itself is installed the same way.

You can have:

* GitLab app = Omnibus + runit + `gitlab-ctl`
* GitLab Runner = separate package + `systemd`

Very common.

---

## Practical mental model

When checking any service on Linux, ask:

### A. Is it OS-managed?

Check:

```bash
systemctl list-units --type=service
```

### B. Is it just a running process?

Check:

```bash
ps aux
pgrep -a postgres
pgrep -a redis
```

### C. Is it containerized?

Check:

```bash
docker ps
podman ps
```

### D. Is it supervised by another framework?

Examples:

* `gitlab-ctl` → GitLab/runit
* `supervisorctl` → supervisord
* `service` / init scripts → SysV init
* custom scripts

---

## Popular ways PostgreSQL can be on a Linux machine

These are the main ones:

1. **apt/yum installed, managed by systemd**
2. **embedded inside another product** like GitLab
3. **Docker/Podman container**
4. **Kubernetes pod/statefulset**
5. **manual/source install with custom scripts**

Same for Redis, Nginx, MySQL, etc.

---

## Very short interview-style answer

> “A database like PostgreSQL can be installed on Linux in multiple ways: as a native OS package managed by systemd, as an embedded dependency inside a product like GitLab Omnibus, inside containers, inside Kubernetes, or via a manual/custom install. `ps aux` shows running processes, while `systemctl` only shows systemd-managed services. GitLab Omnibus commonly ships embedded components under `/opt/gitlab` and manages them with `gitlab-ctl` and runit, so they may not appear as normal `postgresql.service` units.”

## Best commands to tell which model you are dealing with

```bash
ps aux | grep postgres
systemctl list-units --type=service | grep -i postgres
which postgres
ls -l /opt/gitlab/embedded/bin/postgres
docker ps
pgrep -a runsv
```

From your machine, the key clue was:

```bash
/opt/gitlab/embedded/bin/postgres
runsv postgresql
```

That tells you:

* **embedded GitLab PostgreSQL**
* **managed by runit/GitLab tooling**, not plain distro `postgresql.service`

I can also make you a small table: `ps aux vs systemctl vs docker ps vs kubectl` and when to use each.
